# Anonymisation : regex + NER spaCy

Remplace les informations personnelles par des balises `personne_tag`, `organisation_tag`, `email_tag`, `url_tag`, `ip_tag`, `telephone_tag`, `pseudo_tag`, `adresse_tag`.
Les villes, pays et continents sont conservés (non identifiants seuls, utiles à la détection).

In [12]:
import re, time, ipaddress
from pathlib import Path
from collections import Counter
import pandas as pd
import spacy
from tqdm.auto import tqdm   # barre de progression

# Fichiers à anonymiser (toutes les lignes) -> fichiers de sortie, rangés dans un dossier dédié
DOSSIER_ANONYME = Path("data/anonymise")
FICHIERS = {
    "data/train.csv": DOSSIER_ANONYME / "train_anonyme_2.csv",
    "data/test.csv": DOSSIER_ANONYME / "test_anonyme_2.csv",
}

# Balises de remplacement : UN SEUL endroit à modifier (pas de balise pour les lieux : ils ne sont plus masqués)
BALISES = {"PERSONNE": "personne_tag", "ORGANISATION": "organisation_tag", "EMAIL": "email_tag", "URL": "url_tag", "IP": "ip_tag", "TELEPHONE": "telephone_tag", "PSEUDO": "pseudo_tag", "ADRESSE": "adresse_tag"}

# Un span = (début, fin, étiquette) en indices de caractères du texte ORIGINAL
Span = tuple[int, int, str]

/Users/situn/Situu/Projects/depl_ai_cyber/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# REGEX

In [13]:
# --- Motifs regex (compilés une seule fois) ---
RE_EMAIL = re.compile(r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+")
RE_URL = re.compile(r"(?:https?://|ftp://|www\.)[^\s<>\"')\]]+", re.IGNORECASE)
# Candidats IPv4 / IPv6 : on valide ensuite avec le module ipaddress (évite 999.1.1.1, "21:51", etc.)
RE_IPV4 = re.compile(r"(?<![\w.])(?:\d{1,3}\.){3}\d{1,3}(?![\w])")
RE_IPV6 = re.compile(r"(?<![\w:.])(?:[0-9A-Fa-f]{0,4}:){2,7}[0-9A-Fa-f]{0,4}(?![\w:])")
# Téléphone FR (0X XX XX XX XX, +33 X ..., 0033 ...) et international (+CC ...)
RE_TEL_FR = re.compile(r"(?<![\w+])(?:(?:\+|00)33[\s.-]?\(?0?\)?|0)[1-9](?:[\s.-]?\d{2}){4}(?!\d)")
RE_TEL_INTL = re.compile(r"(?<![\w+])\+\d{1,3}[\s.-]?\(?\d{1,4}\)?(?:[\s.-]?\d{2,4}){2,4}(?!\d)")
# Pseudos : @pseudo, "User:Pseudo", "User talk:Pseudo" (on ne masque que le pseudo, pas le préfixe)
RE_AROBASE = re.compile(r"(?<![\w@])@([A-Za-z0-9_]{2,30})")
RE_USER = re.compile(r"\buser(?:[ _]talk)?:\s?([^\s\]|/#<>\[\"]+)", re.IGNORECASE)
# Adresses précises (rue + numéro, code postal) -> [ADRESSE]. On ne masque jamais une simple ville ou un pays.
_TYPES_RUE_FR = r"(?i:rue|avenue|av\.?|boulevard|bd|chemin|impasse|allée|allee|place|route|quai|cours|square|passage)"
_NOM_RUE_FR = r"(?:(?:de|du|des|la|le|les|d'|l')\s*)*[A-ZÀ-Ý][\w'’-]*(?:\s+(?:(?:de|du|des|la|le|les)\s+)*[A-ZÀ-Ý][\w'’-]*){0,3}"
# "12 rue de la Paix" (éventuellement suivi de ", 75002 Paris")
RE_ADRESSE_FR = re.compile(r"\b\d{1,4}(?:\s?(?:bis|ter))?,?\s+" + _TYPES_RUE_FR + r"\s+" + _NOM_RUE_FR
                           + r"(?:,?\s+(?:0[1-9]|[1-8]\d|9[0-5])\d{3}\s+[A-ZÀ-Ý][\w'’-]+)?")
# "221 Baker Street" (nom de rue en majuscule initiale pour limiter les faux positifs)
RE_ADRESSE_EN = re.compile(r"\b\d{1,4}\s+(?:[A-Z][\w'.-]*\s+){1,3}"
                           r"(?:Street|St|Road|Rd|Avenue|Ave|Lane|Ln|Drive|Dr|Boulevard|Blvd|Court|Ct|Way|Place|Pl)\b")
# Codes postaux seuls : FR "75002 Paris", UK "SW1A 1AA", US "NY 10001"
RE_CP_FR = re.compile(r"(?<!\d)(?:0[1-9]|[1-8]\d|9[0-5])\d{3}\s+[A-ZÀ-Ý][\w'’-]+")
RE_CP_UK = re.compile(r"\b[A-Z]{1,2}\d[A-Z\d]?\s?\d[A-Z]{2}\b")
RE_CP_US = re.compile(r"\b[A-Z]{2}\s\d{5}(?:-\d{4})?\b")


def _chiffres(s):
    return sum(ch.isdigit() for ch in s)


def find_regex_spans(text: str) -> list[Span]:
    """Retourne les spans (début, fin, étiquette) détectés par regex.

    Étiquettes : EMAIL, URL, IP, TELEPHONE, PSEUDO, ADRESSE.
    Les spans peuvent se chevaucher entre motifs ; la fusion est faite dans anonymize_A.
    """
    spans: list[Span] = []

    for m in RE_EMAIL.finditer(text):
        spans.append((m.start(), m.end(), "EMAIL"))

    for m in RE_URL.finditer(text):
        # on retire la ponctuation finale collée à l'URL ("... http://x.com.")
        fin = m.end()
        while fin > m.start() and text[fin - 1] in ".,;:!?":
            fin -= 1
        spans.append((m.start(), fin, "URL"))

    for m in RE_IPV4.finditer(text):
        try:
            ipaddress.IPv4Address(m.group())
            spans.append((m.start(), m.end(), "IP"))
        except ValueError:
            pass

    for m in RE_IPV6.finditer(text):
        brut = m.group()
        # exige "::" ou une adresse complète (7 ":") pour ne pas prendre des heures "21:51"
        if "::" not in brut and brut.count(":") != 7:
            continue
        try:
            ipaddress.IPv6Address(brut)
            spans.append((m.start(), m.end(), "IP"))
        except ValueError:
            pass

    for motif in (RE_TEL_FR, RE_TEL_INTL):
        for m in motif.finditer(text):
            if 9 <= _chiffres(m.group()) <= 15:   # filtre les suites de chiffres trop courtes/longues
                spans.append((m.start(), m.end(), "TELEPHONE"))

    for motif in (RE_AROBASE, RE_USER):
        for m in motif.finditer(text):
            spans.append((m.start(1), m.end(1), "PSEUDO"))

    for motif in (RE_ADRESSE_FR, RE_ADRESSE_EN, RE_CP_FR, RE_CP_UK, RE_CP_US):
        for m in motif.finditer(text):
            spans.append((m.start(), m.end(), "ADRESSE"))

    return spans

# Spacy

In [14]:
# Modèle spaCy : on désactive ce dont le NER n'a pas besoin (plus rapide)
nlp = spacy.load("en_core_web_sm", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

# Étiquette spaCy -> étiquette de remplacement.
# NORP (groupes) et GPE / LOC (villes, pays, continents) volontairement absents : on ne les masque pas.
NER_VERS_ETIQUETTE = {"PERSON": "PERSONNE", "ORG": "ORGANISATION"}


def find_ner_spans(doc) -> list[Span]:
    """Retourne les spans (début, fin, étiquette) des entités PERSON et ORG d'un Doc spaCy.

    - les ORG de moins de 3 caractères sont ignorées (souvent du bruit : "FA", "UN"...)
    - NORP, GPE, LOC et les mots toxiques ne sont jamais masqués
    """
    spans: list[Span] = []
    for ent in doc.ents:
        etiquette = NER_VERS_ETIQUETTE.get(ent.label_)
        if etiquette is None:
            continue
        if ent.label_ == "ORG" and len(ent.text.strip()) < 3:
            continue
        spans.append((ent.start_char, ent.end_char, etiquette))
    return spans

# Fusion et anonymisation (A)

In [15]:
def fusionner_spans(spans_regex: list[Span], spans_ner: list[Span]) -> list[Span]:
    """Fusionne regex + NER sans chevauchement. Le regex est prioritaire sur le NER.

    À priorité égale, on garde le span le plus long, puis le plus tôt dans le texte.
    """
    candidats = [(0, s) for s in spans_regex] + [(1, s) for s in spans_ner]
    candidats.sort(key=lambda c: (c[0], -(c[1][1] - c[1][0]), c[1][0]))
    retenus: list[Span] = []
    for _, (d, f, e) in candidats:
        if all(f <= d2 or d >= f2 for d2, f2, _ in retenus):   # aucun chevauchement
            retenus.append((d, f, e))
    return sorted(retenus)


def appliquer_spans(text: str, spans: list[Span]) -> str:
    """Remplace chaque span par sa balise (voir BALISES) en un seul passage sur le texte original."""
    morceaux, curseur = [], 0
    for d, f, e in spans:
        morceaux.append(text[curseur:d])
        morceaux.append(BALISES[e])
        curseur = f
    morceaux.append(text[curseur:])
    return "".join(morceaux)


def anonymize_A(texts: list[str], desc: str = "Anonymisation") -> list[str]:
    """Anonymise une liste de commentaires (regex + NER spaCy) : list[str] -> list[str]."""
    sorties = []
    for text, doc in tqdm(zip(texts, nlp.pipe(texts, batch_size=256)), total=len(texts), desc=desc):
        spans = fusionner_spans(find_regex_spans(text), find_ner_spans(doc))
        sorties.append(appliquer_spans(text, spans))
    return sorties

# Anonymisation de train.csv et test.csv, puis bilan

In [16]:
# --- Anonymisation de TOUTES les lignes de train.csv et test.csv (plusieurs minutes) ---
ETIQUETTES = list(BALISES)
ETIQUETTE_DE = {balise: etiquette for etiquette, balise in BALISES.items()}   # balise -> type
RE_TAG = re.compile("|".join(re.escape(b) for b in BALISES.values()))        # repère les balises du texte anonymisé

bilans = {}
DOSSIER_ANONYME.mkdir(parents=True, exist_ok=True)   # crée le dossier de sortie s'il n'existe pas
for entree, sortie in FICHIERS.items():
    df = pd.read_csv(entree)
    textes = df["comment_text"].fillna("").tolist()

    t0 = time.perf_counter()
    anonymises = anonymize_A(textes, desc=entree)
    duree = time.perf_counter() - t0

    # Sauvegarde : mêmes colonnes que l'entrée, comment_text remplacé par la version anonymisée
    df["comment_text"] = anonymises
    df.to_csv(sortie, index=False)
    print(f"{len(df)} lignes écrites dans {sortie}")

    comptes = Counter(ETIQUETTE_DE[t] for s in anonymises for t in RE_TAG.findall(s))
    bilans[entree] = {"lignes": len(df), "temps (s)": round(duree, 1),
                      "ms / commentaire": round(1000 * duree / len(df), 2),
                      **{e: comptes.get(e, 0) for e in ETIQUETTES}, "TOTAL entités": sum(comptes.values())}

# Bilan : temps + nombre d'entités remplacées par type, pour chaque fichier
display(pd.DataFrame(bilans).T)

data/train.csv: 100%|██████████| 159571/159571 [07:37<00:00, 348.81it/s]


159571 lignes écrites dans data/train_anonyme_2.csv


data/test.csv: 100%|██████████| 153164/153164 [06:49<00:00, 374.00it/s]


153164 lignes écrites dans data/test_anonyme_2.csv


,lignes,temps (s),ms / commentaire,PERSONNE,ORGANISATION,EMAIL,URL,IP,TELEPHONE,PSEUDO,ADRESSE,TOTAL entités
data/train.csv,159571.0,458.1,2.87,117795.0,166238.0,467.0,8032.0,9785.0,79.0,5104.0,224.0,307724.0
data/test.csv,153164.0,409.6,2.67,122052.0,142240.0,537.0,6410.0,4666.0,98.0,1297.0,201.0,277501.0
